In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

def simulate_kyrgyzstan_flood_data(start_date='2018-01-01', end_date='2023-12-31'):
    """
    Simulate hydrological and meteorological data for key flood-prone regions in Kyrgyzstan
    """
    # Convert strings to datetime objects
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.strptime(end_date, '%Y-%m-%d')
    
    # Generate date range
    date_range = [start + timedelta(days=x) for x in range((end-start).days + 1)]
    
    # Define key regions with their characteristics
    regions = {
        # Northern regions
        'Bishkek_City': {'river': 'Ala-Archa', 'elevation': 800, 'flood_threshold': 2.5, 'basin': 'Chu'},
        'Tokmok_Area': {'river': 'Chu', 'elevation': 820, 'flood_threshold': 2.8, 'basin': 'Chu'},
        'Kemin_District': {'river': 'Chon-Kemin', 'elevation': 1300, 'flood_threshold': 1.8, 'basin': 'Chu'},
        'Sokuluk_Valley': {'river': 'Sokuluk', 'elevation': 1100, 'flood_threshold': 1.7, 'basin': 'Chu'},
        
        # Issyk-Kul Basin
        'Karakol_City': {'river': 'Karakol', 'elevation': 1750, 'flood_threshold': 1.9, 'basin': 'Issyk-Kul'},
        'Balykchy_Town': {'river': 'Chu', 'elevation': 1600, 'flood_threshold': 2.0, 'basin': 'Issyk-Kul'},
        'Cholpon-Ata_Area': {'river': 'Cholpon-Ata', 'elevation': 1610, 'flood_threshold': 1.6, 'basin': 'Issyk-Kul'},
        
        # Naryn Basin (Toktogul Dam and surroundings)
        'Toktogul_Dam': {'river': 'Naryn', 'elevation': 870, 'flood_threshold': 4.5, 'basin': 'Naryn'},
        'Naryn_City': {'river': 'Naryn', 'elevation': 2050, 'flood_threshold': 3.2, 'basin': 'Naryn'},
        'Kazarman_Area': {'river': 'Naryn', 'elevation': 1300, 'flood_threshold': 2.8, 'basin': 'Naryn'},
        
        # Talas Basin
        'Talas_City': {'river': 'Talas', 'elevation': 1280, 'flood_threshold': 2.3, 'basin': 'Talas'},
        'Kirov_Dam': {'river': 'Talas', 'elevation': 1090, 'flood_threshold': 3.8, 'basin': 'Talas'},
        
        # Southern regions
        'Osh_City': {'river': 'Ak-Buura', 'elevation': 940, 'flood_threshold': 2.4, 'basin': 'Ferghana'},
        'Jalal-Abad_City': {'river': 'Kogart', 'elevation': 760, 'flood_threshold': 2.1, 'basin': 'Ferghana'},
        'Batken_Area': {'river': 'Syr Darya', 'elevation': 1050, 'flood_threshold': 2.6, 'basin': 'Ferghana'},
        'Arslanbob_Valley': {'river': 'Arslanbob', 'elevation': 1600, 'flood_threshold': 1.7, 'basin': 'Ferghana'},
        
        # Mountain lake areas
        'Song-Kul_Lake': {'river': 'Son-Kul', 'elevation': 3016, 'flood_threshold': 1.2, 'basin': 'Naryn'},
        'Sary-Chelek_Lake': {'river': 'Kojo-Ata', 'elevation': 1940, 'flood_threshold': 1.4, 'basin': 'Western Tien Shan'},
    }
    
    # Create empty dataset
    data = []
    
    # Season parameters (Kyrgyzstan's climate with spring-summer floods)
    seasonal_patterns = {
        1: {'precip_base': 20, 'precip_var': 15, 'snowmelt_factor': 0.2, 'temp_base': -10},  # January
        2: {'precip_base': 25, 'precip_var': 15, 'snowmelt_factor': 0.3, 'temp_base': -8},   # February
        3: {'precip_base': 35, 'precip_var': 20, 'snowmelt_factor': 0.7, 'temp_base': 0},    # March
        4: {'precip_base': 60, 'precip_var': 30, 'snowmelt_factor': 1.3, 'temp_base': 7},    # April
        5: {'precip_base': 70, 'precip_var': 35, 'snowmelt_factor': 1.5, 'temp_base': 12},   # May
        6: {'precip_base': 50, 'precip_var': 30, 'snowmelt_factor': 1.2, 'temp_base': 17},   # June
        7: {'precip_base': 30, 'precip_var': 25, 'snowmelt_factor': 0.8, 'temp_base': 20},   # July
        8: {'precip_base': 20, 'precip_var': 20, 'snowmelt_factor': 0.5, 'temp_base': 19},   # August
        9: {'precip_base': 25, 'precip_var': 20, 'snowmelt_factor': 0.3, 'temp_base': 15},   # September
        10: {'precip_base': 35, 'precip_var': 25, 'snowmelt_factor': 0.2, 'temp_base': 8},   # October
        11: {'precip_base': 30, 'precip_var': 20, 'snowmelt_factor': 0.1, 'temp_base': 0},   # November
        12: {'precip_base': 25, 'precip_var': 15, 'snowmelt_factor': 0.1, 'temp_base': -7}   # December
    }
    
    # Define basin correlation - areas in same basin will have correlated weather patterns
    basin_correlation = {
        'Chu': np.random.normal(0, 1, len(date_range)),
        'Issyk-Kul': np.random.normal(0, 1, len(date_range)),
        'Naryn': np.random.normal(0, 1, len(date_range)),
        'Talas': np.random.normal(0, 1, len(date_range)),
        'Ferghana': np.random.normal(0, 1, len(date_range)),
        'Western Tien Shan': np.random.normal(0, 1, len(date_range))
    }
    
    # Add some cross-correlation between basins
    for i in range(len(date_range)):
        shared_factor = np.random.normal(0, 0.5)  # Common weather pattern across Kyrgyzstan
        for basin in basin_correlation:
            basin_correlation[basin][i] += shared_factor
    
    # Generate extreme events (rare but significant precipitation events)
    extreme_events = []
    num_extreme_events = int((end-start).days * 0.01)  # Approx 3-4 events per year
    
    for _ in range(num_extreme_events):
        event_date_idx = random.randint(0, len(date_range)-1)
        event_duration = random.randint(1, 5)  # 1-5 days
        event_intensity = random.uniform(2.0, 4.0)  # Multiplier for precipitation
        affected_basins = random.sample(list(basin_correlation.keys()), 
                                        random.randint(1, 3))  # 1-3 affected basins
        
        extreme_events.append({
            'start_idx': event_date_idx,
            'duration': event_duration,
            'intensity': event_intensity,
            'affected_basins': affected_basins
        })
    
    # Generate data for each region and date
    for date_idx, date in enumerate(date_range):
        month = date.month
        season_params = seasonal_patterns[month]
        
        # Apply extreme events
        active_extreme_events = [
            event for event in extreme_events 
            if event['start_idx'] <= date_idx < event['start_idx'] + event['duration']
        ]
        
        for region, properties in regions.items():
            basin = properties['basin']
            elevation = properties['elevation']
            
            # Adjust base values by elevation (higher = colder + more precipitation)
            elevation_factor = (elevation - 800) / 1000  # Normalized to have 0 at 800m
            temp_adjustment = -5 * elevation_factor
            precip_adjustment = 10 * elevation_factor
            
            # Base temperature calculation
            temperature = season_params['temp_base'] + temp_adjustment + \
                          3 * basin_correlation[basin][date_idx] + \
                          np.random.normal(0, 2)
            
            # Base precipitation calculation
            precipitation_base = max(0, season_params['precip_base'] + precip_adjustment + \
                                 basin_correlation[basin][date_idx] * season_params['precip_var'] + \
                                 np.random.normal(0, season_params['precip_var']/3))
            
            # Apply extreme events if applicable
            precipitation = precipitation_base
            for event in active_extreme_events:
                if basin in event['affected_basins']:
                    precipitation *= event['intensity']
            
            # Snow accumulation and snowmelt (simplified model)
            snow_depth = 0
            if temperature < 0:
                snow_depth = precipitation  # All precipitation falls as snow
                precipitation = 0
            
            # Calculate snowmelt contribution
            snowmelt = 0
            if temperature > 0 and date_idx > 0:
                # Simplified snowmelt based on temperature and previous snow
                # In a real model, you'd track snow water equivalent
                snowmelt = max(0, min(5 * temperature * season_params['snowmelt_factor'], 50))
            
            # River level calculation - based on precipitation, snowmelt and basin conditions
            base_flow = 0.5 + 0.3 * np.sin(np.pi * month / 6)  # Seasonal base flow
            river_response = 0.02 * precipitation + 0.03 * snowmelt
            river_level = base_flow + river_response + 0.2 * basin_correlation[basin][date_idx]
            
            # Add lag effect (previous days' precipitation affects current river level)
            if date_idx > 0:
                prev_day_effect = data[date_idx-1]['precipitation'] * 0.015
                if date_idx > 1:
                    two_day_effect = data[date_idx-2]['precipitation'] * 0.008
                    river_level += two_day_effect
                river_level += prev_day_effect
            
            # Ensure river level is not negative
            river_level = max(0.1, river_level)
            
            # Flood status based on threshold
            flood_status = 1 if river_level > properties['flood_threshold'] else 0
            
            # Add record to dataset
            data.append({
                'date': date,
                'region': region,
                'river': properties['river'],
                'basin': properties['basin'],
                'elevation': properties['elevation'],
                'temperature': round(temperature, 1),
                'precipitation': round(precipitation, 1),
                'snowmelt': round(snowmelt, 1),
                'river_level': round(river_level, 2),
                'flood_threshold': properties['flood_threshold'],
                'flood_status': flood_status
            })
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Add some soil moisture index (influenced by previous precipitation)
    soil_moisture = []
    for region in regions.keys():
        region_data = df[df['region'] == region].copy()
        region_data = region_data.sort_values('date')
        
        # Initialize soil moisture
        sm = 50  # Start at 50% saturation
        sm_values = []
        
        for _, row in region_data.iterrows():
            # Update soil moisture based on precipitation and temperature
            sm += 0.8 * row['precipitation']  # Increase with precipitation
            sm -= 2 + 0.3 * max(0, row['temperature'])  # Decrease with evaporation
            
            # Constrain to realistic range (0-100%)
            sm = max(0, min(100, sm))
            sm_values.append(sm)
        
        soil_moisture.extend(sm_values)
    
    df['soil_moisture'] = soil_moisture
    
    # Add upstream dam status for relevant regions
    df['upstream_dam_level'] = None
    
    # Define which regions have upstream dams
    dam_influences = {
        'Toktogul_Dam': ['Kazarman_Area', 'Naryn_City'],
        'Kirov_Dam': ['Talas_City'],
    }
    
    # Calculate dam levels and their influence
    for dam, affected_regions in dam_influences.items():
        dam_data = df[df['region'] == dam].copy().sort_values('date')
        dam_data['upstream_dam_level'] = dam_data['river_level'] / dam_data['flood_threshold'] * 100
        
        for region in affected_regions:
            region_dates = df[df['region'] == region]['date'].unique()
            
            for date in region_dates:
                dam_level = dam_data[dam_data['date'] == date]['upstream_dam_level'].values
                if len(dam_level) > 0:
                    df.loc[(df['region'] == region) & (df['date'] == date), 'upstream_dam_level'] = dam_level[0]
    
    return df

# Generate the dataset
flood_data = simulate_kyrgyzstan_flood_data()

# Save to CSV
flood_data.to_csv('kyrgyzstan_flood_data.csv', index=False)
print(f"Generated {len(flood_data)} data points across {flood_data['region'].nunique()} regions")
print(f"Date range: {flood_data['date'].min()} to {flood_data['date'].max()}")
print(f"Number of flood events: {flood_data['flood_status'].sum()}")

Generated 39438 data points across 18 regions
Date range: 2018-01-01 00:00:00 to 2023-12-31 00:00:00
Number of flood events: 16059
